# The Markov-Modulated Asset: Optimal Stopping on a Markov Chain (Julia port)

**A synthesis of optimal stopping and discrete-time finite Markov chains**

An asset has $M$ quality states $S = \{0, \dots, M-1\}$ ($0$ = distressed, $M-1$ = pristine).
Quality evolves each period by a transition matrix $P$. Selling in state $s$ pays $R(s)$;
holding one period costs $c$; sale is forced at horizon $T$. The value function obeys the Bellman recursion

$$V_t(s) = \max\Big(\; R(s)\;,\; -c + \sum_{j \in S} P_{sj}\, V_{t+1}(j) \;\Big), \qquad V_T(s) = R(s).$$

The first branch is **SELL** (stop and collect), the second is **HOLD** (a row of $P$ prices tomorrow).
Because the state space and horizon are finite, backward induction gives the *exact* solution.

Formally, this is a finite-horizon **Markov-modulated optimal stopping** model: a finite Markov chain
modulating an optimal stopping decision. This is a faithful Julia port of the Python notebook, same
matrix, parameters, math, and results.

## Before the math: three layers, kept separate

1. **The model** answers *"what world are we in?"* Five quality levels that hop around by Markov chain
   rules, a price tied to quality, a holding fee each period, and a hard deadline. The goal: pick a
   stopping time $\tau$ to maximize expected profit, $\mathbb{E}[R(S_\tau) - c\,\tau]$.
2. **The rule (the Bellman equation)** answers *"what does optimal mean here?"* At every state and time,
   the best you can do is the better of two numbers: cash now, or the average of tomorrow's best, minus
   the fee.
3. **The recipe (backward induction)** answers *"how do we compute it?"* On the last day the choice is
   forced, so the answer there is known. Walk backward one period at a time and the whole decision table
   fills itself in.

In [ ]:
# Packages: LinearAlgebra and Printf ship with Julia's standard library; Plots needs installation.
# One-time install:
#   import Pkg; Pkg.add(["Plots"])

using LinearAlgebra   # eigen, eigvals, dot, norm, I
using Printf          # @printf formatted output
using Plots           # figures (GR backend by default)

# Save figures into the repo-root figures/ folder. An IJulia notebook's working dir is
# notebooks/, so resolve "../figures"; a headless run from the repo root sees "figures".
const FIGDIR = isdir("figures") ? "figures" : joinpath("..", "figures")

## 1. The Markov environment

`MarkovAsset` validates $P$ (rows sum to 1, non-negative), computes the stationary distribution $\pi$
from the left eigenvector for eigenvalue 1 ($\pi P = \pi$), and measures how fast $P^n$ approaches $\pi$.
The asymptotic rate is the **second-largest eigenvalue modulus** $|\lambda_2|$: the chain forgets its
starting state geometrically, like $|\lambda_2|^n$.

In [ ]:
# A struct plus methods is the idiomatic Julia analogue of a Python class.
struct MarkovAsset
    P::Matrix{Float64}
    M::Int
    function MarkovAsset(P::AbstractMatrix; atol=1e-10)
        P = Float64.(P)
        @assert size(P, 1) == size(P, 2)                     "P must be square"
        @assert all(P .>= -atol)                             "P must have non-negative entries"
        @assert all(isapprox.(sum(P; dims=2), 1.0; atol=atol)) "rows must sum to 1"
        new(P, size(P, 1))
    end
end

# Note: we use the identifier pi (written as the Greek letter) for the stationary vector because it
# matches the math; inside these functions it shadows Base's numeric constant, which we never need here.
function stationary_distribution(a::MarkovAsset)
    F = eigen(permutedims(a.P))  # materialize for a Blas eigen path               # left eigenvectors via P transpose
    idx = argmin(abs.(F.values .- 1.0))     # the eigenvalue nearest 1
    pi = real.(F.vectors[:, idx])
    pi ./= sum(pi)
    @assert all(pi .> 0)                    "chain must be irreducible (pi > 0)"
    return pi
end

# Second-largest eigenvalue modulus: the asymptotic convergence rate.
slem(a::MarkovAsset) = sort(abs.(eigvals(a.P)); rev=true)[2]

# Max-over-rows L2 distance between rows of P^n and pi, for n = 1..n_max.
function convergence_profile(a::MarkovAsset; n_max=50)
    pivec = stationary_distribution(a)
    dists = zeros(n_max)
    Pn = Matrix{Float64}(I, a.M, a.M)
    for n in 1:n_max
        Pn = Pn * a.P
        dists[n] = maximum(norm(Pn[i, :] .- pivec) for i in 1:a.M)
    end
    return dists
end

## 2. The optimal stopping engine

Backward induction: set $V_T = R$, then walk back $t = T-1, \dots, 0$, comparing the immediate reward
against the expected continuation value. The policy matrix records the decision (1 = SELL, 0 = HOLD) at
every $(t, s)$. Where an i.i.d. offer model produces one threshold per time step, here the expectation
$\sum_j P_{sj} V_{t+1}(j)$ depends on the current state, so the rule becomes a boundary in the $(t, s)$ plane.

In [ ]:
struct DynamicStoppingSolver
    asset::MarkovAsset
    R::Vector{Float64}
    c::Float64
    T::Int
    function DynamicStoppingSolver(asset::MarkovAsset, R, c, T)
        R = Float64.(R)
        @assert length(R) == asset.M "R must give one payoff per state"
        @assert c >= 0 && T >= 1
        new(asset, R, Float64(c), Int(T))
    end
end

function solve(s::DynamicStoppingSolver)
    M, P, R, c, T = s.asset.M, s.asset.P, s.R, s.c, s.T
    # Julia is 1-indexed: row t+1 holds the value at time t, for t = 0..T.
    V = zeros(T + 1, M)
    policy = ones(Int, T + 1, M)                 # forced sale at t = T
    V[T + 1, :] .= R
    for t in (T - 1):-1:0
        continuation = -c .+ P * V[t + 2, :]     # value at time t+1 lives in row t+2
        V[t + 1, :] .= max.(R, continuation)
        policy[t + 1, :] .= Int.(R .>= continuation)   # tie-break: sell
    end
    return V, policy
end

## 3. Experiment

The chain has **recovery pressure at the bottom** (distressed assets get fixed up) and **decay pressure at
the top** (pristine assets wear down). Payoffs are $R(s) = 20(s+1)$ in \$k, holding costs $c = 2$ per
period, deadline $T = 12$.

In [ ]:
P = [0.50 0.40 0.10 0.00 0.00;
     0.20 0.45 0.30 0.05 0.00;
     0.05 0.25 0.45 0.20 0.05;
     0.00 0.10 0.30 0.45 0.15;
     0.00 0.05 0.15 0.35 0.45]
R = [20.0, 40.0, 60.0, 80.0, 100.0]   # R(s) = 20(s+1), in $k
c, T = 2.0, 12

asset = MarkovAsset(P)
pivec = stationary_distribution(asset)
println("stationary pi      : ", round.(pivec; digits=4))
@printf("|lambda_2| (SLEM)  : %.4f\n", slem(asset))
@printf("long-run avg R     : %.2f\n", dot(pivec, R))

In [ ]:
solver = DynamicStoppingSolver(asset, R, c, T)
V, policy = solve(solver)
println("V_0 by state: ", round.(V[1, :]; digits=2))
println("policy (rows t = 0..T, 1 = SELL):")
display(policy)

The value function at $t = 0$ is exact: no simulation is needed, because backward induction enumerates
every state and time. Averaging $V_0$ over a uniformly random starting state and comparing against selling
immediately quantifies the value of optimal timing.

In [ ]:
uniform = fill(1 / asset.M, asset.M)
optimal = dot(uniform, V[1, :])
naive   = dot(uniform, R)              # sell at t = 0 regardless of state
@printf("expected profit, optimal rule : %.2f\n", optimal)
@printf("expected profit, sell at t=0  : %.2f\n", naive)
@printf("value of timing               : %+.2f  (%.1f%%)\n", optimal - naive, (optimal/naive - 1)*100)

## 4. Convergence of $P^n$ to $\pi$

How far is the worst-case row of $P^n$ from the stationary distribution? The dashed reference shows the
geometric envelope $|\lambda_2|^n$, and the match is exact: $|\lambda_2| = 0.680$ is the true forgetting
rate. Memory of the initial state is effectively gone within ~10 periods, which is why stopping decisions
only matter early.

In [ ]:
dists = convergence_profile(asset; n_max=40)
lam2 = slem(asset)
n = 1:40

plot(n, dists; yscale=:log10, marker=:circle, ms=3, lw=1.8, color=:steelblue,
     label="max_i ||(P^n)_i. - pi||_2",
     xlabel="n (matrix power)", ylabel="L2 distance to pi (log scale)",
     title="Convergence of P^n to the stationary distribution", legend=:topright)
plot!(n, dists[1] .* lam2 .^ (n .- 1); ls=:dash, lw=1.5, color=:firebrick,
      label="geometric rate |lambda_2|^n, |lambda_2| = $(round(lam2; digits=3))")
savefig(joinpath(FIGDIR, "convergence.png"))

## 5. The decision boundary

Time on the x-axis, state on the y-axis. Orange = SELL, blue = HOLD. The optimal rule is **monotone**:
SELL iff $s \ge s^*(t)$, with $s^* = 3$ for $t \le 7$, $s^* = 2$ for $8 \le t \le 11$, and $s^* = 0$ at the
forced-sale deadline. The boundary steps *down* as the deadline nears, because patience loses value when
few recovery chances remain.

In [ ]:
grid = permutedims(policy)              # M x (T+1): rows = state, cols = time
heatmap(0:T, 0:(asset.M - 1), grid;
        color=cgrad([:steelblue, :sandybrown]), clims=(0, 1), colorbar=false,
        xlabel="time t", ylabel="quality state s",
        title="Optimal policy: SELL (orange) vs HOLD (blue)",
        xticks=0:T, yticks=0:(asset.M - 1))
for t in 0:T, s in 0:(asset.M - 1)
    sell = policy[t + 1, s + 1] == 1
    annotate!(t, s, text(sell ? "S" : "H", 9, sell ? :black : :white))
end
savefig(joinpath(FIGDIR, "policy_heatmap.png"))
current()

## 6. Interpretation and limitations

**Answer.** Following the threshold rule, an asset of uniformly random quality is worth $\approx 71.1$ (\$k),
an 18% gain over selling immediately. High states liquidate at once; low states hold and wait for recovery.

**Division of labor.** The Markov chain predicts the environment ($P$, $\pi$, $|\lambda_2|$) but cannot decide anything;
the optimal-stopping rule turns those predictions into actions but is blind to tomorrow once offers aren't i.i.d.

**Binding limitation.** $P$ is assumed known and time-homogeneous; a misspecified or shifting $P$ moves the
boundary and erodes the edge.

**Out of reach.** If buyers react strategically to our selling rule, no single-agent optimum exists. That is
a game-theory question, unanswerable with any amount of data or compute.

## Verification against the Python reference

The cell below checks the Julia results against the values the Python notebook produced. Every line should
print `true`. Any `false` is flagged with the computed-versus-reference numbers so you can trace it.

In [ ]:
ref_pi     = [0.1445, 0.2849, 0.3051, 0.1869, 0.0787]
ref_lam2   = 0.6795
ref_piR    = 55.41
ref_V0     = [53.35, 57.25, 64.68, 80.0, 100.0]
ref_opt    = 71.06
ref_naive  = 60.00
ref_cutoff = [3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 0]

cutoff = [findfirst(==(1), policy[t + 1, :]) - 1 for t in 0:T]   # smallest SELL state s*(t)

@printf("pi match      : %s\n", all(isapprox.(pivec, ref_pi; atol=1e-3)))
@printf("|lambda_2| ok : %s  (%.4f vs %.4f)\n", isapprox(slem(asset), ref_lam2; atol=1e-3), slem(asset), ref_lam2)
@printf("pi.R ok       : %s  (%.2f vs %.2f)\n", isapprox(dot(pivec, R), ref_piR; atol=1e-2), dot(pivec, R), ref_piR)
@printf("V_0 ok        : %s\n", all(isapprox.(round.(V[1, :]; digits=2), ref_V0; atol=1e-2)))
@printf("optimal ok    : %s  (%.2f vs %.2f)\n", isapprox(optimal, ref_opt; atol=1e-2), optimal, ref_opt)
@printf("naive ok      : %s  (%.2f vs %.2f)\n", isapprox(naive, ref_naive; atol=1e-2), naive, ref_naive)
@printf("cutoff ok     : %s  (%s)\n", cutoff == ref_cutoff, cutoff)